# Introduction to Modeling Libraries

**Estimated time:** 45–60 minutes
**Prerequisites:** numpy_intermediate, pandas_intermediate

## Learning Goals

| # | Topic |
|---|-------|
| 1 | Interfacing pandas with model code — `to_numpy()`, feature matrices, train/test split |
| 2 | Patsy — R-style formula syntax for design matrices |
| 3 | statsmodels — OLS regression, GLMs, and autoregressive time series |
| 4 | scikit-learn — `fit`/`predict` API, logistic regression, cross-validation |

---

### Quick Reference

```python
# pandas → model input
X = df[features].to_numpy()
X_enc = pd.get_dummies(df, columns=['line'], drop_first=True)
train, test = df.iloc[:n_train], df.iloc[n_train:]

# patsy
from patsy import dmatrices
y, X = dmatrices('loss_ratio ~ premium + C(line)', data=df, return_type='dataframe')

# statsmodels
import statsmodels.formula.api as smf
res = smf.ols('loss_ratio ~ premium + C(line)', data=df).fit()
print(res.summary()); res.predict(new_data)

# scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
model = LogisticRegression().fit(X_train, y_train)
model.predict(X_test); model.predict_proba(X_test)
cross_val_score(model, X, y, cv=5, scoring='accuracy')
```

In [ ]:
import numpy as np
import pandas as pd

print('numpy', np.__version__)
print('pandas', pd.__version__)

In [ ]:
# Insurance policy dataset — used throughout the notebook
rng = np.random.default_rng(42)
n = 300

lines      = rng.choice(['Auto', 'Property', 'Liability'], size=n)
years_ifs  = rng.integers(1, 15, size=n)          # years in force
premium    = rng.integers(800, 8_000, size=n).astype(float)
prior_clms = rng.integers(0, 4, size=n)            # prior claims count

# Loss ratio: higher for newer policies and prior claims
noise      = rng.normal(0, 0.08, size=n)
base_lr    = {'Auto': 0.65, 'Property': 0.58, 'Liability': 0.54}
loss_ratio = np.array([base_lr[l] for l in lines])              + 0.03 * prior_clms              - 0.005 * years_ifs              + noise
loss_ratio = np.clip(loss_ratio, 0.20, 1.30)

# Binary target: did the policy have a claim this year?
claim_prob = 0.10 + 0.06 * prior_clms / 3
had_claim  = rng.binomial(1, np.clip(claim_prob, 0, 1)).astype(int)

df = pd.DataFrame({
    'line':       lines,
    'years_ifs':  years_ifs,
    'premium':    premium,
    'prior_clms': prior_clms,
    'loss_ratio': loss_ratio.round(4),
    'had_claim':  had_claim,
})
print(df.shape)
print(df.head())

---
## Section 1 — Interfacing pandas with Model Code

Most modeling libraries (statsmodels, scikit-learn) work with NumPy arrays or plain numeric
DataFrames — not with arbitrary pandas objects containing strings or mixed dtypes.

| Task | Method |
|------|--------|
| DataFrame → NumPy array | `df[cols].to_numpy()` |
| Encode categorical columns | `pd.get_dummies(df, columns=[...], drop_first=True)` |
| Train / test split (time-ordered) | `df.iloc[:n_train]` / `df.iloc[n_train:]` |
| Train / test split (random) | `sklearn.model_selection.train_test_split` |
| Separate features and target | `X = df.drop(columns='target'); y = df['target']` |

**Rule of thumb:** get your data into a fully numeric DataFrame first, then call `.to_numpy()`
only when a library requires it — many accept DataFrames directly.

In [ ]:
# Select numeric features + encode categorical
feature_cols = ['years_ifs', 'premium', 'prior_clms', 'line']
target_col   = 'loss_ratio'

df_enc = pd.get_dummies(df[feature_cols + [target_col]],
                        columns=['line'], drop_first=True)
print('Encoded columns:', df_enc.columns.tolist())
print(df_enc.head(3))

# Separate X and y
X = df_enc.drop(columns=target_col)
y = df_enc[target_col]

print(f'\nX shape: {X.shape}   y shape: {y.shape}')

# to_numpy() for libraries that require plain arrays
X_arr = X.to_numpy()
print('X as NumPy:', X_arr.shape, X_arr.dtype)

In [ ]:
from sklearn.model_selection import train_test_split

# Random 80/20 split (stratified on line for balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'Train: {len(X_train)} rows   Test: {len(X_test)} rows')

# Time-ordered split (first 240 = train, last 60 = test)
n_train = 240
X_train_t = X.iloc[:n_train]
X_test_t  = X.iloc[n_train:]
y_train_t = y.iloc[:n_train]
y_test_t  = y.iloc[n_train:]
print(f'Time-ordered split: {len(X_train_t)} train / {len(X_test_t)} test')

# Check for class balance in the binary target
print('\nhad_claim distribution:')
print(df['had_claim'].value_counts(normalize=True).round(3))

In [ ]:
# EXERCISE 1:
# a) Build a feature DataFrame using columns: years_ifs, premium, prior_clms, line
#    Encode 'line' with get_dummies(drop_first=True) and set had_claim as target y
# b) Print dtypes of the encoded DataFrame — confirm all are numeric
# c) Do an 80/20 train/test split (random_state=0) on (X, y)
# d) Print the mean of y_train and y_test — they should be similar if split is balanced
# YOUR CODE HERE

---
## Section 2 — Patsy: Formula-Based Design Matrices

Patsy lets you describe a model with an R-style formula string instead of manually
building NumPy arrays. It handles intercepts, categorical encoding, and transformations.

```python
from patsy import dmatrices
y, X = dmatrices('response ~ predictor1 + predictor2', data=df, return_type='dataframe')
```

| Syntax | Meaning |
|--------|---------|
| `y ~ x1 + x2` | Intercept + x1 + x2 |
| `y ~ x1 + x2 + 0` | No intercept |
| `C(col)` | Treat numeric column as categorical |
| `np.log(x)` | Apply transformation inline |
| `x1:x2` | Interaction term only |
| `x1 * x2` | Main effects + interaction (`x1 + x2 + x1:x2`) |

Patsy `DesignMatrix` objects carry column names and factor metadata, making them easy to
inspect and pass to statsmodels.

In [ ]:
try:
    from patsy import dmatrices, dmatrix

    # Basic formula: loss_ratio predicted by premium and line (categorical)
    y_pat, X_pat = dmatrices(
        'loss_ratio ~ premium + C(line)',
        data=df,
        return_type='dataframe',
    )
    print('Response (y) shape:', y_pat.shape)
    print('Design matrix (X) shape:', X_pat.shape)
    print('X columns:', X_pat.columns.tolist())
    print(X_pat.head(3))

    # No intercept: add + 0
    _, X_no_int = dmatrices('loss_ratio ~ premium + C(line) + 0', data=df,
                             return_type='dataframe')
    print('\nNo-intercept columns:', X_no_int.columns.tolist())

except ImportError:
    print('patsy not installed — run: pip install patsy')

In [ ]:
try:
    from patsy import dmatrices

    # Inline transformations using numpy functions
    y_t, X_t = dmatrices(
        'loss_ratio ~ np.log(premium) + prior_clms + years_ifs + C(line)',
        data=df,
        return_type='dataframe',
    )
    print('Transformed columns:', X_t.columns.tolist())
    print(X_t[['np.log(premium)', 'prior_clms']].describe().round(3))

    # Interaction term: prior_clms × years_ifs
    _, X_int = dmatrices(
        'loss_ratio ~ prior_clms * years_ifs + C(line)',
        data=df, return_type='dataframe',
    )
    print('\nWith interaction:', X_int.columns.tolist())

except ImportError:
    print('patsy not installed — run: pip install patsy')

In [ ]:
# EXERCISE 2:
# a) Use dmatrices to build a design matrix for:
#    loss_ratio ~ np.log(premium) + prior_clms + C(line) + 0  (no intercept)
#    Print the resulting X column names and the first 3 rows
# b) Build a formula with an interaction term between prior_clms and years_ifs
#    (use prior_clms:years_ifs for interaction only, or prior_clms*years_ifs for full)
# c) Print the shape of X for both formulas and explain why they differ
try:
    from patsy import dmatrices
    # YOUR CODE HERE
except ImportError:
    print('pip install patsy')

---
## Section 3 — statsmodels: Classical Statistical Models

statsmodels implements frequentist linear models, GLMs, ANOVA, and time series.
It has two API styles — the **formula API** (recommended) accepts pandas DataFrames directly.

| Function / Class | Purpose |
|-----------------|---------|
| `smf.ols(formula, data).fit()` | Ordinary least squares regression |
| `smf.glm(formula, data, family).fit()` | Generalized linear model |
| `res.summary()` | Full regression table (coefs, p-values, R²) |
| `res.params` | Fitted coefficients as a Series |
| `res.predict(new_data)` | Predictions on new data |
| `AutoReg(endog, lags).fit()` | Autoregressive time series model |

**Interpretation tip:** `res.pvalues < 0.05` marks statistically significant predictors.
Use `res.rsquared_adj` (not `rsquared`) to compare models with different numbers of predictors.

In [ ]:
try:
    import statsmodels.formula.api as smf

    # OLS: predict loss_ratio from premium, prior claims, years in force, and line
    model = smf.ols(
        'loss_ratio ~ np.log(premium) + prior_clms + years_ifs + C(line)',
        data=df,
    )
    res = model.fit()

    # Summary: coefficients, std errors, t-stats, p-values, R²
    print(res.summary().tables[1])   # just the coefficients table
    print(f'\nAdj R²: {res.rsquared_adj:.4f}')
    print(f'AIC:    {res.aic:.2f}')

    # Significant predictors
    sig = res.pvalues[res.pvalues < 0.05]
    print('\nSignificant at 5%:')
    print(sig.round(4))

    # Predict on new policies
    new_policies = pd.DataFrame({
        'premium':    [1_500, 4_000, 8_000],
        'prior_clms': [0, 1, 3],
        'years_ifs':  [10, 5, 2],
        'line':       ['Auto', 'Property', 'Liability'],
    })
    preds = res.predict(new_policies)
    print('\nPredicted loss ratios:', preds.round(4).tolist())

except ImportError:
    print('statsmodels not installed — run: pip install statsmodels')

In [ ]:
try:
    import statsmodels.api as sm
    from statsmodels.tsa.ar_model import AutoReg

    # Build a monthly aggregate loss series
    idx = pd.date_range('2018-01', periods=60, freq='ME')
    rng2 = np.random.default_rng(7)
    monthly_lr = pd.Series(
        0.62 + np.cumsum(rng2.normal(0, 0.01, 60)) + rng2.normal(0, 0.03, 60),
        index=idx, name='loss_ratio',
    )

    # Fit AR(3) model
    ar_model = AutoReg(monthly_lr, lags=3)
    ar_res   = ar_model.fit()

    print('AR(3) coefficients:')
    print(ar_res.params.round(4))
    print(f'AIC: {ar_res.aic:.2f}')

    # Forecast next 6 months
    forecast = ar_res.predict(start=len(monthly_lr), end=len(monthly_lr) + 5)
    print('\n6-month forecast:')
    print(forecast.round(4).to_string())

except ImportError:
    print('statsmodels not installed — run: pip install statsmodels')

In [ ]:
# EXERCISE 3:
# a) Fit an OLS model predicting loss_ratio with only prior_clms and years_ifs (no line).
#    Print the Adj R² and compare it to the full model above — does adding 'line' help?
# b) Use res.params to extract the coefficient on prior_clms and interpret it:
#    "A one-unit increase in prior claims is associated with a ___ change in loss ratio"
# c) Predict loss_ratio for a new policy: Auto, 3 years in force, 2 prior claims, $2500 premium
# d) (Optional) Fit a GLM with a Gaussian family on the same formula as in the demo:
#    smf.glm(formula, data, family=sm.families.Gaussian()).fit()
try:
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    # YOUR CODE HERE
except ImportError:
    print('pip install statsmodels')

---
## Section 4 — scikit-learn: Machine Learning Workflow

scikit-learn provides a **consistent API** across all model types:
1. Create a model object with hyperparameters
2. `.fit(X_train, y_train)` — learn from data
3. `.predict(X_test)` — generate point predictions
4. `.predict_proba(X_test)` — class probabilities (classifiers only)
5. `cross_val_score(model, X, y, cv=5)` — k-fold cross-validation

| Class / Function | Purpose |
|-----------------|---------|
| `LogisticRegression(C, max_iter)` | Binary/multiclass classification |
| `LinearRegression()` | OLS regression (no p-values) |
| `SimpleImputer(strategy)` | Fill missing values before modeling |
| `train_test_split(X, y, test_size)` | Holdout split |
| `cross_val_score(model, X, y, cv, scoring)` | K-fold CV scores |
| `accuracy_score`, `roc_auc_score` | Classification metrics |

**Key difference from statsmodels:** scikit-learn focuses on **prediction accuracy** and
does not provide p-values or confidence intervals — use statsmodels for inference.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Encode and prepare
df_enc2 = pd.get_dummies(df[['years_ifs', 'premium', 'prior_clms', 'line', 'had_claim']],
                          columns=['line'], drop_first=True)
X2 = df_enc2.drop(columns='had_claim').to_numpy()
y2 = df_enc2['had_claim'].to_numpy()

# Scale features (LogisticRegression converges faster with standardized inputs)
scaler = StandardScaler()
X_tr, X_te, y_tr, y_te = train_test_split(X2, y2, test_size=0.20, random_state=42,
                                            stratify=y2)
X_tr_s = scaler.fit_transform(X_tr)   # fit on train only
X_te_s = scaler.transform(X_te)       # apply same scaling to test

# Fit logistic regression
clf = LogisticRegression(C=1.0, max_iter=500, random_state=42)
clf.fit(X_tr_s, y_tr)

y_pred  = clf.predict(X_te_s)
y_proba = clf.predict_proba(X_te_s)[:, 1]

print(f'Accuracy: {accuracy_score(y_te, y_pred):.3f}')
print(f'ROC-AUC:  {roc_auc_score(y_te, y_proba):.3f}')

# Coefficients → which features matter most?
feat_names = df_enc2.drop(columns='had_claim').columns
coef_df = pd.Series(clf.coef_[0], index=feat_names).sort_values(key=abs, ascending=False)
print('\nTop feature coefficients:')
print(coef_df.round(3))

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Cross-validation: 5-fold CV for logistic regression on had_claim
df_enc3 = pd.get_dummies(df[['years_ifs', 'premium', 'prior_clms', 'line', 'had_claim']],
                          columns=['line'], drop_first=True)
X_cv = df_enc3.drop(columns='had_claim').to_numpy()
y_cv = df_enc3['had_claim'].to_numpy()

# Pipeline: impute → scale → model (imputation needed if data has NaN)
pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=500, random_state=42),
)

cv_scores = cross_val_score(pipeline, X_cv, y_cv, cv=5, scoring='roc_auc')
print('5-fold CV ROC-AUC scores:', cv_scores.round(3))
print(f'Mean: {cv_scores.mean():.3f}  ±  {cv_scores.std():.3f}')

# Linear regression with CV for continuous loss_ratio target
X_lr = df_enc3.drop(columns=['had_claim']).to_numpy()
y_lr = df['loss_ratio'].to_numpy()

lr_pipeline = make_pipeline(SimpleImputer(strategy='median'),
                              StandardScaler(), LinearRegression())
r2_scores = cross_val_score(lr_pipeline, X_lr, y_lr, cv=5, scoring='r2')
print(f'\nLinear regression 5-fold R²: {r2_scores.mean():.3f}  ±  {r2_scores.std():.3f}')

In [ ]:
# EXERCISE 4:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Using the had_claim target and features: years_ifs, premium, prior_clms, encoded line
df_ex = pd.get_dummies(df[['years_ifs', 'premium', 'prior_clms', 'line', 'had_claim']],
                        columns=['line'], drop_first=True)
X_ex = df_ex.drop(columns='had_claim').to_numpy()
y_ex = df_ex['had_claim'].to_numpy()

# a) Fit a LogisticRegression with C=0.1 (stronger regularization) in a pipeline
#    with StandardScaler; print its 5-fold CV accuracy score
# b) Fit a second model with C=10 (weaker regularization); compare CV accuracy
# c) Print the confusion matrix for the better model on a held-out 20% test set
#    Hint: from sklearn.metrics import confusion_matrix
# YOUR CODE HERE

---
## Wrap-Up Cheat Sheet

| Topic | Key takeaway |
|-------|--------------|
| pandas → model | `get_dummies(drop_first=True)` + `to_numpy()` before passing to sklearn |
| Train/test split | `train_test_split(X, y, test_size=0.2, stratify=y)` for classification |
| Patsy | `dmatrices('y ~ x + C(cat)', data=df)` handles intercept + encoding automatically |
| Patsy transforms | Embed numpy in formulas: `np.log(x)`, `x1:x2` for interaction |
| statsmodels OLS | `smf.ols(formula, data).fit()` — gives p-values, confidence intervals, R² |
| statsmodels AutoReg | `AutoReg(series, lags).fit()` — AR(p) for time series |
| sklearn fit/predict | `model.fit(X_train, y_train)` → `model.predict(X_test)` |
| sklearn pipeline | `make_pipeline(SimpleImputer(), StandardScaler(), model)` — prevents data leakage |
| Cross-validation | `cross_val_score(pipeline, X, y, cv=5)` — better than single train/test split |
| statsmodels vs sklearn | statsmodels = inference (p-values); sklearn = prediction (accuracy, AUC) |